# Lab 10：變分自編碼器 VAE（Colab 版）

**Colab 版說明**（張珀銀 2026/10/16）

課本程式寫於 TensorFlow 2.0 時代，在現在的 Colab 會出錯。這一版只改必要的地方，改過的儲存格開頭都標了「【Colab 版修改】」，其餘和課本相同。改了：
- 課本 Lab10 是外部專案 [TF2-VAE](https://github.com/KUASWoodyLIN/TF2-VAE)（作者 Woody Lin，MIT 授權），原本要自己 clone 再照 README 執行。這一本把步驟寫好，在 Colab 裡 clone、修正、訓練、測試
- 第一格「Colab 準備」：舊版 Keras 相容設定
- `utils/callbacks.py` 的 `np.Inf` 改 `np.inf`（NumPy 2 已移除）
- `test.py` 讀模型的資料夾 `logs_vae_1/` 改 `logs_vae/`（`train.py` 存在 `logs_vae/`，原本對不上）

**使用方式**：先在「執行階段 → 變更執行階段類型」選 GPU，再選「執行階段 → 全部執行」。MNIST 資料由 tfds 從 Google 主機下載，很快，不用另外上傳。

In [ ]:
# 【Colab 版修改】Colab 準備：每次開啟或重新啟動後，最先執行這一格
import os
# 課本寫於 TF 2.0 時代，改用 Colab 內建的舊版 Keras（tf_keras）；下面用 !python 執行的程式也沿用這個設定
os.environ['TF_USE_LEGACY_KERAS'] = '1'

# 下載課本的專案並進入資料夾（重複執行這一格時不會再做一次）
if os.path.basename(os.getcwd()) != 'TF2-VAE':
    if not os.path.exists('TF2-VAE'):
        !git clone -q https://github.com/KUASWoodyLIN/TF2-VAE
    %cd TF2-VAE

### 修正兩處新版不相容

In [ ]:
# 【Colab 版修改】專案程式只改兩處，其餘和課本相同
def fix(path, old, new):
    # 把 path 檔案裡的 old 換成 new；已經換過就跳過，內容和預期不同就停下來
    text = open(path, encoding='utf-8').read()
    if old in text:
        open(path, 'w', encoding='utf-8').write(text.replace(old, new))
    elif new not in text:
        raise RuntimeError(path + ' 的內容和預期不同，請告訴老師')

fix('utils/callbacks.py', 'np.Inf', 'np.inf')  # NumPy 2 已移除 np.Inf
fix('test.py', 'logs_vae_1/', 'logs_vae/')  # train.py 把模型存在 logs_vae/，test.py 原本讀 logs_vae_1/

### 確認用的是舊版 Keras（應顯示 tf_keras）

In [ ]:
!python -c "import tensorflow as tf; print('TensorFlow', tf.__version__, '；Keras 用的是', tf.keras.__name__.split('.')[0])"

### 訓練：`train.py`（MNIST，20 個 epoch，最好的解碼器存到 `logs_vae/models/`）

In [ ]:
!python train.py

### 測試：`test.py` 從二維潛在空間取 15×15 個點，用解碼器畫出數字

In [ ]:
!python test.py
from IPython.display import Image
Image('output.png')

### TensorBoard：看損失曲線與每個 epoch 的解碼結果

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs_vae